# Receipt OCR — QA Pipeline

End-to-end: synthetic data generation → QA fine-tuning → ONNX export → inference.

| Stage | Description |
|-------|-------------|
| 1 | Generate synthetic fuel receipts in SQuAD v2 format |
| 2 | Fine-tune `deepset/minilm-uncased-squad2` (~90 MB) |
| 3 | Export to ONNX, quantize to INT8, convert to FP16 |
| 4 | Inference: PaddleOCR → sliding-window context → QA extraction |

In [44]:
!apt-get install -y -q tesseract-ocr
!pip install -q pytesseract Pillow datasets evaluate transformers torch \
    safetensors onnx onnxruntime onnxruntime-tools onnxconverter-common

## 1. Synthetic Dataset Generation

Design goals vs the naive approach:
- **Structured templates** (5 receipt layouts) — no random field shuffling
- **Position-tracked spans** — answer offsets recorded during construction, zero alignment failures
- **Realistic OCR noise** — char confusion table, digit transposition, space merge/split, punct sub
- **Questions aligned with inference** — training and deployed question strings are identical

In [8]:
import json
import random
import uuid
import re
import os
from datetime import datetime, timedelta

NUM_EXAMPLES   = 10_000
OUTPUT_FILE    = "synthetic_receipt_ocr_squad.json"
OCR_ERROR_RATE = 0.07

In [9]:
# --- station name pools ---
indian_station_names = [
    "IndianOil Petrol Pump", "IOCL Pump - Kisan Seva Kendra", "HPCL Retail Outlet",
    "Hindustan Petroleum", "Bharat Petroleum", "BPCL Fuel Station", "Reliance Petroleum",
    "Nayara Energy Fuel Station", "Essar Oil Petrol Pump", "Shell India Pump",
    "Adani Gas CNG Station", "GAIL Gas Station", "Sharma Fuel Centre", "Gupta Filling Station",
    "Singh & Sons Petroleum", "Patel Fuels", "Rajeshwari Service Station",
    "Krishna Filling Point", "Om Sai Ram Fuels", "Sri Venkateswara Fuels",
    "Jain Brothers Pump", "Aggarwal Petroleum", "Verma Filling Centre",
    "Chaudhary Fuels", "Malik Auto Fuels", "Liberty Fuel Station",
    "National Petroleum", "Highway Fuel Point", "City Gas Stop",
    "Ganga Filling Station", "Yamuna Petroleum", "Sunrise Fuels",
    "Lucky Filling Station", "Star Fuels", "Diamond Petroleum",
    "Golden Fuel Point", "Express Fuel Stop", "Balaji Petroleum",
    "Murugan Fuels", "Annapurna Filling Station", "Friends Auto Service",
    "Modern Fuel Centre", "New India Petroleum", "United Fuels",
    "Premier Petroleum", "Standard Fuels", "Royal Fuel Station",
    "Welcome Filling Station", "Evergreen Fuels",
] + [f"Generic Indian Fuel Stop {i}" for i in range(150)]

na_uk_station_names = [
    "Shell", "Exxon", "Mobil", "Chevron", "Texaco", "BP", "Amoco",
    "Sunoco", "Marathon", "Valero", "Phillips 66", "Conoco", "76",
    "Circle K Fuel", "7-Eleven Fuel", "Costco Gasoline", "Sam's Club Gas",
    "Wawa Fuel", "Sheetz Gas", "QuikTrip Fuel", "Kwik Trip Gas",
    "Pilot Flying J", "Love's Travel Stop", "Petro-Canada", "Esso",
    "Husky Energy", "Chevron Canada", "Shell Canada", "Co-op Gas Bar",
    "Irving Oil", "Ultramar", "Pioneer Energy", "Canadian Tire Gas+",
    "Tesco Petrol Filling Station", "Sainsbury's Petrol Station", "Asda Petrol",
    "Morrisons Petrol", "Shell UK", "BP UK", "Esso UK", "Texaco UK",
    "Gulf Oil", "Jet Petroleum", "Murco Petroleum", "Applegreen PLC",
    "Rontec Roadside Retail", "MFG - Motor Fuel Group", "GULF", "Racetrac", "Speedway",
    "Maverik", "Sinclair", "Arco", "Casey's General Store", "Buc-ee's",
    "GasBuddy Partner", "UK Fuels Station", "Keyfuels Site", "Fast Fuels",
    "Admiral", "Go Gas", "SuperGas", "FuelZone", "Petro Express",
] + [f"Generic NA/UK Fuel {i}" for i in range(100)]

other_station_names = [
    "TotalEnergies", "ENI Station", "Repsol Service Station", "Petrobras",
    "Sinopec Gas Station", "PetroChina Fuel", "Gazprom Neft", "Lukoil Station",
    "Rosneft Fuel", "Equinor Energy Station", "OMV Filling Station",
    "PKN Orlen", "MOL Group Station", "Cepsa Estacion de Servicio", "Galp Energia",
    "Petronas Fuel Station", "PTT Station (Thailand)", "Caltex", "Z Energy (NZ)",
    "Ampol (Australia)", "Engen Petroleum (SA)", "Sasol Fuel (SA)",
    "YPF (Argentina)", "Ecopetrol (Colombia)", "ADNOC Distribution (UAE)",
    "Emirates General Petroleum (Emarat)", "ENOC/EPPCO (Dubai)",
    "Aral Tankstelle (Germany)", "Avia Tankstelle", "Orlen Deutschland",
    "Neste (Finland)", "St1 Station", "Circle K Europe", "Preem (Sweden)",
    "Itochu Enex (Japan)", "Eneos (Japan)", "Cosmo Oil (Japan)", "Idemitsu (Japan)",
    "SK Energy (S. Korea)", "GS Caltex (S. Korea)", "S-Oil (S. Korea)",
    "Pertamina (Indonesia)", "Petron (Philippines)", "Saudi Aramco Station",
    "Pemex (Mexico)", "Gaz'Up (France)", "Neste MY (Malaysia)", "Socar (Turkey)",
] + [f"Global Fuel Point {i}" for i in range(50)]

# address components
street_types_india  = ["Road", "Rd", "Marg", "Path", "Street", "St", "Main Road", "Cross Road", "Lane", "Bypass", "Highway", "NH"]
street_types_na_uk  = ["Street", "St", "Avenue", "Ave", "Road", "Rd", "Drive", "Dr", "Lane", "Ln", "Boulevard", "Blvd", "Court", "Ct", "Place", "Pl", "Way", "Terrace", "Highway", "Hwy"]
street_types_other  = ["Rue", "Avenida", "Strasse", "Via", "Calle", "Road", "Street", "Jalan", "Platz", "Ring", "Gasse"]

localities_india = ["Nehru Nagar", "Gandhi Colony", "Shivaji Park", "Sector 15", "Model Town", "Ashok Vihar", "Rajendra Place", "Patel Nagar", "Vivek Nagar", "Anandpuri", "Subhash Chowk", "Lajpat Nagar"]
localities_na_uk = ["Oak Street", "Maple Avenue", "Main Street", "High Street", "Park Road", "Station Road", "Church Lane", "Elm Drive", "Cedar Court", "Victoria Place", "King's Way", "Queen's Terrace"]
localities_other = ["Bahnhofstrasse", "Rue de la Paix", "Gran Via", "Corso Italia", "Paseo de la Reforma", "Orchard Road", "Shibuya Crossing Area", "Hauptplatz", "Jl. Sudirman", "Friedrichstrasse"]

cities_india = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Kolkata", "Pune", "Ahmedabad", "Jaipur", "Lucknow", "Kanpur", "Nagpur", "Indore", "Thane", "Bhopal", "Visakhapatnam", "Patna", "Vadodara", "Ghaziabad", "Ludhiana"]
cities_na    = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Toronto", "Montreal", "Vancouver", "Calgary", "Edmonton", "Ottawa", "Mississauga", "Winnipeg", "Hamilton", "Quebec City"]
cities_uk    = ["London", "Birmingham", "Manchester", "Glasgow", "Liverpool", "Bristol", "Sheffield", "Leeds", "Edinburgh", "Leicester", "Coventry", "Nottingham", "Newcastle", "Brighton", "Southampton"]
cities_other = ["Paris", "Berlin", "Rome", "Madrid", "Tokyo", "Beijing", "Shanghai", "Moscow", "Sao Paulo", "Mexico City", "Cairo", "Istanbul", "Seoul", "Jakarta", "Sydney", "Dubai", "Singapore", "Buenos Aires", "Bangkok", "Amsterdam"]

states_india = ["MH", "DL", "KA", "TG", "TN", "WB", "GJ", "RJ", "UP", "MP", "PB", "HR", "BR", "AP"]
states_na    = ["NY", "CA", "IL", "TX", "AZ", "PA", "FL", "OH", "GA", "NC", "ON", "QC", "BC", "AB", "MB", "SK"]
states_uk    = ["", "London", "West Midlands", "Greater Manchester", "Merseyside", "Scotland", "Wales", "N. Ireland", "Surrey", "Kent", "Essex"]
states_other = ["", "Ile-de-France", "Berlin", "Lazio", "Madrid", "Tokyo", "Beijing", "Moscow", "Sao Paulo", "CDMX", "Dubai", "Singapore"]

pincode_pattern_india  = r"[1-8]\d{5}"
zipcode_pattern_na     = r"\d{5}(-\d{4})?|[A-Z]\d[A-Z] \d[A-Z]\d"
postcode_pattern_uk    = r"[A-Z]{1,2}\d[A-Z\d]? \d[A-Z]{2}"
postcode_pattern_other = [r"\d{4,5}", r"\d{2}-\d{3}", r"[A-Z]{1,2}\d{1,4}", r"\d{3} \d{2}"]

phone_formats_india = ["+91-##########", "+91 ##########", "0##########", "##########", "(0XX) ########", "XXXXX XXXXX"]
phone_formats_na    = ["(###) ###-####", "###-###-####", "1-###-###-####", "+1##########", "###.###.####"]
phone_formats_uk    = ["0#### ######", "02# #### ####", "01### ######", "+44 ##########", "07#########"]
phone_formats_other = ["+## ## ### ####", "+### ### ### ###", "0# #### ####", "## #### ####", "(##) ####-####", "#### ### ###"]

fuel_types = [
    "Petrol", "Gasoline", "Unleaded", "Regular Unleaded", "Premium Unleaded",
    "Super Unleaded", "Diesel", "Bio-Diesel", "Diesel B7", "Premium Diesel",
    "CNG", "Compressed Natural Gas", "LPG", "AutoGas", "E10", "E85", "Ethanol Blend",
    "MS (Motor Spirit)", "HSD (High Speed Diesel)", "Power Petrol", "Speed Diesel",
    "Regular Gas", "Midgrade Gas", "Premium Gas", "V-Power", "Synergy Supreme+", "Ultimate Unleaded",
] * 10

currencies = {
    "India":        ("INR", "\u20b9", "Rs."),
    "USA":          ("USD", "$", "USD"),
    "Canada":       ("CAD", "$", "CAD"),
    "UK":           ("GBP", "\u00a3", "GBP"),
    "Eurozone":     ("EUR", "\u20ac", "EUR"),
    "Australia":    ("AUD", "$", "AUD"),
    "Singapore":    ("SGD", "$", "SGD"),
    "UAE":          ("AED", "AED", "Dhs"),
    "Japan":        ("JPY", "\u00a5", "JPY"),
    "China":        ("CNY", "\u00a5", "CNY"),
    "Germany":      ("EUR", "\u20ac", "EUR"),
    "France":       ("EUR", "\u20ac", "EUR"),
    "Mexico":       ("MXN", "$", "MXN"),
    "Brazil":       ("BRL", "R$", "BRL"),
    "South Africa": ("ZAR", "R", "ZAR"),
}
units = {
    "India": "Litre", "USA": "Gallon", "Canada": "Litre", "UK": "Litre",
    "Eurozone": "Litre", "Australia": "Litre", "Singapore": "Litre",
    "UAE": "Litre", "Japan": "Litre", "China": "Litre",
    "Germany": "Litre", "France": "Litre",
    "Mexico": "Litro", "Brazil": "Litro", "South Africa": "Litre",
}
unit_abbr = {"Litre": "Ltr", "Gallon": "Gal", "Litro": "Lt", "Kilogram": "Kg"}

# OCR character confusion table (research-based patterns)
OCR_CONFUSION = {
    '0': ['O', 'D'],   'O': ['0', 'Q'],
    '1': ['l', 'I'],   'l': ['1', 'I'],   'I': ['1', 'l'],
    '2': ['Z'],        'Z': ['2', '7'],
    '3': ['8'],        '8': ['B', '3'],
    '5': ['S'],        'S': ['5'],
    '6': ['G'],        'G': ['6', 'C'],
    '9': ['q'],        'q': ['9'],
    'B': ['8', 'R'],
    '.': [','],        ',': ['.'],
    ':': [';'],        ';': [':'],
    '-': ['_'],        '_': ['-'],
}

RECEIPT_TEMPLATES = ['vertical', 'compact', 'table', 'minimal', 'verbose']

# Questions aligned exactly with Section 4 inference
questions_map = {
    "station_name": "What is the name of the fuel station?",
    "station_addr": "What is the address of the fuel station?",
    "date":         "What is the date of the transaction?",
    "currency":     "What is the currency symbol or code?",
    "fuel_type":    "What type of fuel was purchased?",
    "unit_price":   "What is the price per unit of fuel?",
    "volume":       "What volume of fuel was purchased?",
    "total_amount": "What is the total amount paid?",
}

In [10]:
def generate_random_date(start="2020-01-01", end="2025-04-20"):
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end,   "%Y-%m-%d")
    d = s + timedelta(days=random.randint(0, (e - s).days))
    fmt  = random.choice(["%d/%m/%Y", "%m/%d/%Y", "%Y-%m-%d", "%d-%b-%Y", "%b %d, %Y",
                           "%d.%m.%Y", "%Y.%m.%d", "%d %B %Y", "%d-%m-%y", "%m/%d/%y"])
    tfmt = random.choice(["%H:%M", "%H:%M:%S", "%I:%M %p", ""])
    return d.strftime(fmt) + (" " + d.strftime(tfmt) if tfmt else "")


def generate_pincode(pattern):
    if pattern == pincode_pattern_india:
        return str(random.randint(100000, 899999))
    if pattern == zipcode_pattern_na:
        if random.random() < 0.7:
            z = str(random.randint(10000, 99999))
            if random.random() < 0.1:
                z += "-" + str(random.randint(1000, 9999))
            return z
        al = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        return (f"{random.choice('ABCDEFGHIJKLMNOPRSTUVWXY')}{random.randint(0,9)}{random.choice(al)} "
                f"{random.randint(0,9)}{random.choice(al)}{random.randint(0,9)}")
    if pattern == postcode_pattern_uk:
        al = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        p1 = random.choice(al) + (random.choice(al) if random.random() < 0.5 else "")
        p2 = str(random.randint(0, 9)) + (random.choice(al + "0123456789") if random.random() < 0.5 else "")
        return f"{p1}{p2} {random.randint(0,9)}{random.choice(al)}{random.choice(al)}"
    pt = random.choice(pattern)
    if pt == r"\d{4,5}":      return str(random.randint(1000, 99999))
    if pt == r"\d{2}-\d{3}": return f"{random.randint(10,99)}-{random.randint(100,999)}"
    if pt == r"[A-Z]{1,2}\d{1,4}":
        al = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        p  = random.choice(al) + (random.choice(al) if random.random() < 0.3 else "")
        n  = random.randint(1, 4)
        return p + str(random.randint(0, 10**n - 1)).zfill(n)
    if pt == r"\d{3} \d{2}": return f"{random.randint(100,999)} {random.randint(10,99)}"
    return "PC000"


def generate_phone(formats):
    fmt = random.choice(formats)
    while "#" in fmt: fmt = fmt.replace("#", str(random.randint(0, 9)), 1)
    while "X" in fmt: fmt = fmt.replace("X", str(random.randint(1, 9)), 1)
    return fmt


def apply_ocr_noise(text: str, rate: float = OCR_ERROR_RATE) -> str:
    """Realistic OCR noise: char confusion, case flip, space errors, punct sub, digit shift."""
    if not text or rate <= 0:
        return text
    chars = list(text)
    i = 0
    while i < len(chars):
        if random.random() < rate:
            c     = chars[i]
            ntype = random.choices(
                ['confusion', 'case', 'space_ins', 'space_del', 'punct', 'digit_adj'],
                weights=[0.30, 0.12, 0.13, 0.13, 0.12, 0.20]
            )[0]
            if   ntype == 'confusion' and c in OCR_CONFUSION:
                chars[i] = random.choice(OCR_CONFUSION[c])
            elif ntype == 'case' and c.isalpha():
                chars[i] = c.swapcase()
            elif ntype == 'space_ins' and c not in (' ', '\n'):
                chars.insert(i + 1, ' '); i += 1
            elif ntype == 'space_del' and c == ' ':
                chars.pop(i); continue
            elif ntype == 'punct':
                subs = {'.': ',', ',': '.', ':': ';', ';': ':', '-': '_', '_': '-'}
                if c in subs: chars[i] = subs[c]
            elif ntype == 'digit_adj' and c.isdigit():
                chars[i] = str((int(c) + random.choice([-1, 1])) % 10)
        i += 1
    result = ''.join(chars)
    # digit transposition (separate pass)
    if random.random() < rate * 0.6:
        m = re.search(r'\d{2,}', result)
        if m:
            digs = list(m.group(0))
            j    = random.randint(0, len(digs) - 2)
            digs[j], digs[j + 1] = digs[j + 1], digs[j]
            result = result[:m.start()] + ''.join(digs) + result[m.end():]
    return result


def build_receipt(gt: dict) -> tuple:
    """Build a noisy structured receipt from clean ground truth values.

    Returns (context, spans) where spans[field] = (noisy_text, start_idx).
    Spans are recorded during construction so alignment is always exact.
    """
    template = random.choice(RECEIPT_TEMPLATES)
    parts, spans, pos = [], {}, 0
    recorded = set()

    def emit(text: str, field: str = None):
        nonlocal pos
        if field and field not in recorded:
            spans[field] = (text, pos)
            recorded.add(field)
        parts.append(text)
        pos += len(text)

    def deco(text: str):
        emit(apply_ocr_noise(text, rate=0.02))

    nv    = {k: apply_ocr_noise(gt[k]) for k in questions_map if k in gt}
    cs    = apply_ocr_noise(gt.get('currency', ''), rate=0.03)
    phone = apply_ocr_noise(gt.get('phone', ''), rate=0.05)
    usym  = gt.get('unit_sym', 'Ltr')
    uname = gt.get('unit_name', 'Litre')

    if template == 'vertical':
        emit(nv['station_name'] + '\n', 'station_name')
        emit(nv['station_addr'] + '\n', 'station_addr')
        deco(phone + '\n\n')
        deco(random.choice(['Date', 'DATE', 'Dt']) + ': ')
        emit(nv['date'] + '\n', 'date')
        deco('\n')
        deco(random.choice(['Fuel Type', 'Product', 'Item']) + ' : ')
        emit(nv['fuel_type'] + '\n', 'fuel_type')
        deco('Volume    : ')
        emit(nv['volume'] + ' ' + usym + '\n', 'volume')
        deco('Rate      : ')
        emit(cs, 'currency')
        emit(nv['unit_price'] + '/' + usym + '\n', 'unit_price')
        deco(random.choice(['Amount', 'Subtotal', 'Net']) + '    : ')
        emit(cs, 'currency')
        emit(nv['total_amount'] + '\n', 'total_amount')
        deco('\n')
        deco(random.choice(['NET TOTAL', 'TOTAL', 'Grand Total']) + ' : ')
        deco(cs + apply_ocr_noise(gt['total_amount'], rate=0.02) + '\n')
        if random.random() < 0.5:
            deco('Receipt No: ' + str(random.randint(10000, 99999)) + '\n')
        if random.random() < 0.4:
            deco('Payment: ' + random.choice(['Cash', 'Card', 'UPI']) + '\n')
        deco(random.choice(['Thank You!', 'Visit Again!', '']) + '\n')

    elif template == 'compact':
        emit(nv['station_name'], 'station_name')
        deco(' | ')
        emit(nv['station_addr'], 'station_addr')
        deco(' | ' + random.choice(['Tel', 'Ph']) + ': ' + phone + '\n')
        deco(random.choice(['Date', 'DATE']) + ': ')
        emit(nv['date'] + '\n', 'date')
        emit(nv['fuel_type'], 'fuel_type')
        deco(' ')
        emit(nv['volume'] + usym, 'volume')
        deco(' @ ')
        emit(cs, 'currency')
        emit(nv['unit_price'] + '/' + usym + '\n', 'unit_price')
        deco(random.choice(['TOTAL', 'Total']) + ': ')
        emit(cs, 'currency')
        emit(nv['total_amount'] + '\n', 'total_amount')

    elif template == 'table':
        emit(nv['station_name'] + '\n', 'station_name')
        emit(nv['station_addr'] + '\n', 'station_addr')
        deco(phone + '\n\n')
        deco('DATE: ')
        emit(nv['date'] + '\n', 'date')
        deco('\n')
        deco('ITEM                 QTY       RATE      AMOUNT\n')
        deco('-' * 50 + '\n')
        emit(nv['fuel_type'] + '  ', 'fuel_type')
        emit(nv['volume'], 'volume')
        deco('  ')
        emit(nv['unit_price'], 'unit_price')
        deco('    ')
        emit(nv['total_amount'], 'total_amount')
        deco('\n' + '-' * 50 + '\n')
        deco('Currency: ')
        emit(cs + '\n', 'currency')
        deco('TOTAL: ' + cs + apply_ocr_noise(gt['total_amount'], rate=0.02) + '\n')

    elif template == 'minimal':
        emit(nv['station_name'] + '\n', 'station_name')
        emit(nv['station_addr'] + '\n', 'station_addr')
        emit(nv['date'] + '\n', 'date')
        emit(nv['fuel_type'] + '\n', 'fuel_type')
        emit(nv['volume'], 'volume')
        deco(' x ')
        emit(nv['unit_price'], 'unit_price')
        deco(' = ')
        emit(cs, 'currency')
        emit(nv['total_amount'] + '\n', 'total_amount')
        deco('Total: ' + cs + apply_ocr_noise(gt['total_amount'], rate=0.02) + '\n')

    else:  # verbose
        deco('*' * 30 + '\n')
        deco('     FUEL PURCHASE RECEIPT\n')
        deco('*' * 30 + '\n')
        deco('Station  : ')
        emit(nv['station_name'] + '\n', 'station_name')
        deco('Address  : ')
        emit(nv['station_addr'] + '\n', 'station_addr')
        deco('Contact  : ' + phone + '\n\n')
        deco('Txn Date : ')
        emit(nv['date'] + '\n', 'date')
        deco('\n')
        deco('Product  : ')
        emit(nv['fuel_type'] + '\n', 'fuel_type')
        deco('Volume   : ')
        emit(nv['volume'], 'volume')
        deco(' ' + uname + '\n')
        deco('Unit Rate: ')
        emit(cs, 'currency')
        deco(' ')
        emit(nv['unit_price'] + '/' + usym + '\n', 'unit_price')
        deco('-' * 30 + '\n')
        deco('Net Amt  : ')
        emit(cs, 'currency')
        deco(' ')
        emit(nv['total_amount'] + '\n', 'total_amount')
        deco('-' * 30 + '\n')
        if random.random() < 0.4:
            deco('Mode     : ' + random.choice(['Cash', 'Card', 'UPI']) + '\n')
        if random.random() < 0.3:
            deco('Receipt  : ' + str(random.randint(10000, 99999)) + '\n')
        deco('*' * 30 + '\n')

    return ''.join(parts), spans

In [11]:
def generate_squad_dataset(num_examples: int = NUM_EXAMPLES, output_file: str = OUTPUT_FILE):
    dataset = {"version": "v2.0", "data": []}
    all_ids = set()
    n_ok    = 0

    for i in range(num_examples):
        if i % (num_examples // 10) == 0 and i > 0:
            print(f"Generated {i}/{num_examples} ({n_ok} valid)...")

        rv = random.random()
        if rv < 0.50:
            region    = "India"
            name_pool = indian_station_names
            street_t, locs = street_types_india, localities_india
            cities_p, states_p = cities_india, states_india
            pinpat, phones = pincode_pattern_india, phone_formats_india
        elif rv < 0.80:
            region    = random.choice(["USA", "Canada", "UK"])
            name_pool = na_uk_station_names
            street_t, locs = street_types_na_uk, localities_na_uk
            if region == "UK":
                cities_p, states_p = cities_uk, states_uk
                pinpat, phones = postcode_pattern_uk, phone_formats_uk
            else:
                cities_p, states_p = cities_na, states_na
                pinpat, phones = zipcode_pattern_na, phone_formats_na
        else:
            region    = random.choice(list(set(currencies.keys()) - {"India", "USA", "Canada", "UK"}))
            name_pool = other_station_names
            street_t, locs = street_types_other, localities_other
            cities_p, states_p = cities_other, states_other
            pinpat, phones = postcode_pattern_other, phone_formats_other

        curr_code, curr_sym, curr_lbl = currencies.get(region, currencies["USA"])
        unit_name = units.get(region, "Litre")
        unit_sym  = unit_abbr.get(unit_name, unit_name)
        fuel_type = random.choice(fuel_types)

        if "Diesel" in fuel_type:             price_rng = (0.9, 2.0) if unit_name == "Litre" else (3.5, 6.0)
        elif "CNG" in fuel_type or "LPG" in fuel_type:
            price_rng = (0.5, 1.5)
            unit_name = random.choice(["Litre", "Kilogram"])
            unit_sym  = unit_abbr.get(unit_name, unit_name)
        else:                                 price_rng = (1.0, 2.5) if unit_name == "Litre" else (3.0, 7.0)

        dec            = random.choice([2, 3])
        unit_price_val = round(random.uniform(*price_rng), dec)
        volume_val     = round(random.uniform(5.0, 80.0), dec)
        total_val      = round(unit_price_val * volume_val, 2)
        city           = random.choice(cities_p)
        state          = random.choice(states_p)

        gt = {
            "station_name": random.choice(name_pool),
            "station_addr": ", ".join(filter(None, [
                f"{random.randint(1,999)} {random.choice(locs)} {random.choice(street_t)}",
                (f"{city}, {state}" if state else city),
                generate_pincode(pinpat),
            ])),
            "date":         generate_random_date(),
            "fuel_type":    fuel_type,
            "unit_price":   f"{unit_price_val:.{dec}f}",
            "volume":       f"{volume_val:.{dec}f}",
            "total_amount": f"{total_val:.2f}",
            "currency":     random.choice([curr_sym, curr_code, curr_lbl]),
            "phone":        generate_phone(phones),
            "unit_sym":     unit_sym,
            "unit_name":    unit_name,
        }

        context, spans = build_receipt(gt)

        qas = []
        for field, question in questions_map.items():
            if field not in spans:
                continue
            ans_text, ans_start = spans[field]
            if context[ans_start: ans_start + len(ans_text)] != ans_text:
                continue  # sanity check (should never fail with position tracking)
            q_id = str(uuid.uuid4())
            while q_id in all_ids: q_id = str(uuid.uuid4())
            all_ids.add(q_id)
            qas.append({
                "question":      question,
                "id":            q_id,
                "answers":       [{"text": ans_text, "answer_start": ans_start}],
                "is_impossible": False,
            })

        if len(qas) < 4:
            continue

        n_ok += 1
        title = f"Receipts_{region}"
        para  = {"context": context, "qas": qas}
        for item in dataset["data"]:
            if item["title"] == title:
                item["paragraphs"].append(para)
                break
        else:
            dataset["data"].append({"title": title, "paragraphs": [para]})

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    print(f"Saved {output_file}  ({n_ok}/{num_examples} valid examples)")


generate_squad_dataset()

Generated 1000/10000 (1000 valid)...
Generated 2000/10000 (2000 valid)...
Generated 3000/10000 (3000 valid)...
Generated 4000/10000 (4000 valid)...
Generated 5000/10000 (5000 valid)...
Generated 6000/10000 (6000 valid)...
Generated 7000/10000 (7000 valid)...
Generated 8000/10000 (8000 valid)...
Generated 9000/10000 (9000 valid)...
Saved synthetic_receipt_ocr_squad.json  (10000/10000 valid examples)


## 2. Fine-Tune QA Model

Fine-tunes `deepset/minilm-uncased-squad2` on the generated dataset.

In [12]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    TrainingArguments, Trainer, default_data_collator,
)
import numpy as np

MODEL_NAME    = "deepset/minilm-uncased-squad2"
DATA_PATH     = OUTPUT_FILE
OUTPUT_DIR    = "ocr-qa-finetuned"
BATCH_SIZE    = 128    # T4 15 GB — safe for MiniLM-L12 at seq_len=384 with fp16
MAX_LEN       = 384
DOC_STRIDE    = 128
NUM_EPOCHS    = 3
LEARNING_RATE = 3e-5
WEIGHT_DECAY  = 0.01
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
raw  = load_dataset("json", data_files=DATA_PATH, field="data")["train"]
flat = []
for ex in raw:
    for para in ex["paragraphs"]:
        ctx = para["context"]
        for qa in para["qas"]:
            if not qa["answers"]:
                continue
            ans = qa["answers"][0]
            if ans["text"] and ans["text"] in ctx:
                flat.append({
                    "id":       f"ex-{len(flat)}",
                    "question": qa["question"],
                    "context":  ctx,
                    "answers":  [ans],
                })

split = Dataset.from_list(flat).train_test_split(test_size=0.1, seed=42)
ds    = DatasetDict({"train": split["train"], "validation": split["test"]})
print(f"Train: {len(ds['train'])}, Val: {len(ds['validation'])}")

Generating train split: 0 examples [00:00, ? examples/s]

Train: 72000, Val: 8000


In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_features(examples):
    tok = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", max_length=MAX_LEN,
        stride=DOC_STRIDE, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding="max_length",
    )
    sample_map = tok.pop("overflow_to_sample_mapping")
    offsets    = tok.pop("offset_mapping")
    starts, ends = [], []
    for i, offset in enumerate(offsets):
        ans     = examples["answers"][sample_map[i]][0]
        sc, ec  = ans["answer_start"], ans["answer_start"] + len(ans["text"])
        seq     = tok.sequence_ids(i)
        ts      = seq.index(1)
        te      = len(seq) - 1 - seq[::-1].index(1)
        if not (offset[ts][0] <= sc and offset[te][1] >= ec):
            starts.append(0); ends.append(0)
        else:
            while ts < len(offset) and offset[ts][0] <= sc: ts += 1
            starts.append(ts - 1)
            while offset[te][1] >= ec: te -= 1
            ends.append(te + 1)
    tok["start_positions"] = starts
    tok["end_positions"]   = ends
    return tok

tok_ds = ds.map(prepare_features, batched=True,
                remove_columns=ds["train"].column_names, num_proc=4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Map (num_proc=4):   0%|          | 0/72000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/8000 [00:00<?, ? examples/s]

In [15]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME, ignore_mismatched_sizes=True)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        do_train=True, do_eval=True,
        fp16=True,                    
        eval_strategy="steps", eval_steps=200,
        logging_steps=50, save_steps=200, save_total_limit=3,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        report_to="none",
    ),
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    processing_class=tokenizer,
    data_collator=default_data_collator,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss,Validation Loss
200,0.060212,0.044224
400,0.046392,0.039637
600,0.039473,0.036755
800,0.036132,0.036678
1000,0.044853,0.036162
1200,0.036330,0.035826
1400,0.033235,0.035025
1600,0.036964,0.035053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ocr-qa-finetuned


## 3. ONNX Export & Quantization

Produces three model artifacts:
- `model_fp32.onnx` — baseline
- `model_int8.onnx` — dynamic INT8 (~4× smaller)
- `model_fp16.onnx` — FP16 for GPU/NPU deployment

In [17]:
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxconverter_common import convert_float_to_float16
import torch, onnx

ONNX_MODEL_DIR = OUTPUT_DIR
ONNX_FP32      = "model_fp32.onnx"
ONNX_INT8      = "model_int8.onnx"
ONNX_FP16      = "model_fp16.onnx"
ONNX_SEQ_LEN   = 512

model_onnx     = AutoModelForQuestionAnswering.from_pretrained(ONNX_MODEL_DIR)
tokenizer_onnx = AutoTokenizer.from_pretrained(ONNX_MODEL_DIR)
model_onnx.eval()

dummy = tokenizer_onnx(
    "What is the total amount paid?", "Total: $36.32",
    return_tensors="pt", padding="max_length",
    max_length=ONNX_SEQ_LEN, truncation=True,
)

torch.onnx.export(
    model_onnx,
    (dummy["input_ids"], dummy["attention_mask"], dummy.get("token_type_ids")),
    ONNX_FP32,
    dynamo=False,                 # force TorchScript exporter (torch 2.x compat)
    input_names=["input_ids", "attention_mask", "token_type_ids"],
    output_names=["start_logits", "end_logits"],
    dynamic_axes={
        "input_ids":      {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "token_type_ids": {0: "batch", 1: "seq"},
        "start_logits":   {0: "batch"},
        "end_logits":     {0: "batch"},
    },
    opset_version=14,
)
print(f"FP32: {ONNX_FP32}")

quantize_dynamic(ONNX_FP32, ONNX_INT8, weight_type=QuantType.QInt8)
print(f"INT8: {ONNX_INT8}")

onnx.save(convert_float_to_float16(onnx.load(ONNX_FP32)), ONNX_FP16)
print(f"FP16: {ONNX_FP16}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_53453/421241471.py:21: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:171: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean

FP32: model_fp32.onnx


INT8: model_int8.onnx


/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 9.999999960041972e-13 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:63: UserWarning: the float32 number -inf will be truncated to -10000.0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.549071314612462e-10 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -5.6876590548426975e-08 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 9.846630888432628e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -6.76328326676412e-08 will be tru

FP16: model_fp16.onnx


## 4. Inference

Both variants share the OCR utilities below.

- **4a — Transformers pipeline**: use during development; requires `transformers`.
- **4b — ONNX runtime**: production-grade, only `onnxruntime` + `paddleocr` needed.

In [58]:
import pytesseract
from PIL import Image

def run_ocr(image_path: str) -> list:
    text = pytesseract.image_to_string(Image.open(image_path))
    return [line.strip() for line in text.split('\n') if line.strip()]

def make_context_windows(lines: list, max_len: int = 384, stride: int = 128) -> list:
    tokens  = " ".join(lines).split()
    windows, start = [], 0
    while start < len(tokens):
        windows.append(" ".join(tokens[start:start + max_len]))
        if start + max_len >= len(tokens):
            break
        start += (max_len - stride)
    return windows

QUESTIONS = {
    "station_name": "What is the name of the fuel station?",
    "station_addr": "What is the address of the fuel station?",
    "date":         "What is the date of the transaction?",
    "currency":     "What is the currency symbol or code?",
    "fuel_type":    "What type of fuel was purchased?",
    "unit_price":   "What is the price per unit of fuel?",
    "volume":       "What volume of fuel was purchased?",
    "total_amount": "What is the total amount paid?",
}
CONFIDENCE_THRESHOLD = 0.4

In [59]:
import re

def _clean_field(field: str, value: str) -> str:
    if value is None:
        return value

    # numeric fields: extract first decimal number
    if field in ('unit_price', 'volume', 'total_amount'):
        m = re.search(r'\d+\.\d+', value)
        if m: return m.group(0)
        m = re.search(r'\d+', value)
        return m.group(0) if m else value

    # date: extract date + optional time, drop trailing text
    if field == 'date':
        m = re.search(
            r'\d{1,2}[/\-.]\ ?\d{1,2}[/\-.]\ ?\d{2,4}'
            r'(?:\s+\d{1,2}:\d{2}(?::\d{2})?(?:\s*[AaPp][Mm])?)?',
            value)
        return m.group(0).strip() if m else value

    # text fields: strip trailing tokens that are pure uppercase
    # 4+ chars with no digits or punctuation — heuristic for label bleed
    parts = value.split()
    while len(parts) > 1 and re.match(r'^[A-Z]{4,}$', parts[-1]):
        parts.pop()

    # station_addr: strip leading standalone number (pump number)
    if field == 'station_addr' and parts and re.match(r'^\d+$', parts[0]):
        parts.pop(0)

    return ' '.join(parts) if parts else value


def postprocess(results: dict) -> dict:
    return {k: {**v, 'value': _clean_field(k, v['value'])} for k, v in results.items()}

In [60]:
# --- 4a: Transformers pipeline (development) ---
from transformers import pipeline
import torch, json

def extract_info_transformers(image_path: str, model_path: str = OUTPUT_DIR) -> dict:
    device = 0 if torch.cuda.is_available() else -1
    qa     = pipeline("question-answering", model=model_path, tokenizer=model_path, device=device)
    ctxs   = make_context_windows(run_ocr(image_path))
    out    = {}
    for key, question in QUESTIONS.items():
        best = {"score": 0.0, "answer": ""}
        for ctx in ctxs:
            r = qa(question=question, context=ctx)
            if r["score"] > best["score"]:
                best = r
        out[key] = {
            "value":      best["answer"].strip() if best["score"] >= CONFIDENCE_THRESHOLD else None,
            "confidence": round(best["score"], 3),
        }
    return postprocess(out)

# info = extract_info_transformers("test.jpg")
# print(json.dumps(info, indent=2))

In [61]:
import base64, io, json, os
from PIL import Image

B64_IMAGE = "..."

# decode → PIL → save as test.jpg
img = Image.open(io.BytesIO(base64.b64decode(B64_IMAGE))).convert('RGB')
img.save('test.jpg', 'JPEG', quality=95)
print(f"Saved test.jpg  {img.size}")

# run inference
info = extract_info_transformers('test.jpg')
print(json.dumps(info, indent=2))

Saved test.jpg  (1024, 744)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{
  "station_name": {
    "value": "COASTAL FUEL #114",
    "confidence": 1.0
  },
  "station_addr": {
    "value": "123 BAYSIDE AVE, SEASIDE, CA 93955",
    "confidence": 1.0
  },
  "date": {
    "value": "91/24/2024 10:45:32 AM",
    "confidence": 0.999
  },
  "currency": {
    "value": "$",
    "confidence": 0.997
  },
  "fuel_type": {
    "value": "GALLONS",
    "confidence": 0.885
  },
  "unit_price": {
    "value": "4.299",
    "confidence": 0.965
  },
  "volume": {
    "value": "14.250",
    "confidence": 0.496
  },
  "total_amount": {
    "value": "61.26",
    "confidence": 0.699
  }
}


In [52]:
# --- 4b: ONNX runtime + custom tokenizer (production, no transformers dep) ---
import onnxruntime as ort
import numpy as np
import json, re


class SelfTokenizer:
    """Dependency-free WordPiece tokenizer compatible with BERT vocab files."""

    def __init__(self, vocab_file: str, do_lower_case: bool = True):
        self.do_lower_case = do_lower_case
        self.vocab         = self._load_vocab(vocab_file)
        self.inv_vocab     = {v: k for k, v in self.vocab.items()}
        self.unk = "[UNK]"; self.sep = "[SEP]"; self.cls = "[CLS]"; self.pad = "[PAD]"

    def _load_vocab(self, path):
        with open(path, encoding="utf-8") as f:
            return {tok.strip(): i for i, tok in enumerate(f)}

    def _tokenize(self, text):
        if self.do_lower_case:
            text = text.lower()
        tokens = []
        for word in text.strip().split():
            buf = ""
            for ch in word:
                if re.match(r"\w", ch): buf += ch
                else:
                    if buf: tokens.append(buf); buf = ""
                    tokens.append(ch)
            if buf: tokens.append(buf)
        return tokens

    def _wordpiece(self, token):
        if token in self.vocab:
            return [token]
        pieces, start = [], 0
        while start < len(token):
            end, cur = len(token), None
            while start < end:
                sub = token[start:end] if start == 0 else "##" + token[start:end]
                if sub in self.vocab: cur = sub; break
                end -= 1
            if cur is None: return [self.unk]
            pieces.append(cur); start = end
        return pieces

    def tokenize(self, text):
        tokens = []
        for t in self._tokenize(text):
            tokens.extend(self._wordpiece(t))
        return tokens

    def convert_tokens_to_ids(self, tokens):
        return [self.vocab.get(t, self.vocab[self.unk]) for t in tokens]

    def __call__(self, question: str, context: str, max_length: int = 512) -> dict:
        q_toks = self.tokenize(question)
        c_toks = self.tokenize(context)
        tokens = [self.cls] + q_toks + [self.sep] + c_toks + [self.sep]
        if len(tokens) > max_length:
            tokens = tokens[:max_length - 1] + [self.sep]
        input_ids      = self.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)
        n_q            = min(len(q_toks) + 2, max_length)
        token_type_ids = [0] * n_q + [1] * (len(input_ids) - n_q)
        pad            = max_length - len(input_ids)
        input_ids      += [self.vocab[self.pad]] * pad
        attention_mask += [0] * pad
        token_type_ids += [0] * pad
        return {
            "input_ids":      np.array([input_ids],      dtype=np.int64),
            "attention_mask": np.array([attention_mask], dtype=np.int64),
            "token_type_ids": np.array([token_type_ids], dtype=np.int64),
        }


class ONNXQuestionAnswering:
    def __init__(self, model_path: str, vocab_file: str):
        self.session      = ort.InferenceSession(model_path)
        self.tokenizer    = SelfTokenizer(vocab_file)
        self._input_names = {inp.name for inp in self.session.get_inputs()}

    def __call__(self, question: str, context: str) -> dict:
        inputs = self.tokenizer(question, context)
        feed   = {k: v for k, v in inputs.items() if k in self._input_names}
        s_log, e_log = self.session.run(["start_logits", "end_logits"], feed)
        s, e = int(np.argmax(s_log[0])), int(np.argmax(e_log[0]))
        if s > e: s, e = e, s
        ids    = inputs["input_ids"][0][s:e + 1].tolist()
        answer = "".join(self.tokenizer.inv_vocab.get(i, "") for i in ids).replace("##", "")
        score  = float(1 / (1 + np.exp(-(float(np.max(s_log)) + float(np.max(e_log))))))
        return {"answer": answer.strip(), "score": score}


def extract_info_onnx(
    image_path: str,
    model_path: str = ONNX_INT8,
    vocab_file: str = None,
) -> dict:
    if vocab_file is None:
        vocab_file = os.path.join(ONNX_MODEL_DIR, "vocab.txt")
    qa   = ONNXQuestionAnswering(model_path, vocab_file)
    ctxs = make_context_windows(run_ocr(image_path))
    out  = {}
    for key, question in QUESTIONS.items():
        best = {"score": 0.0, "answer": ""}
        for ctx in ctxs:
            r = qa(question=question, context=ctx)
            if r["score"] > best["score"]:
                best = r
        out[key] = {
            "value":      best["answer"] if best["score"] >= CONFIDENCE_THRESHOLD else None,
            "confidence": round(best["score"], 3),
        }
    return out

# info = extract_info_onnx("receipt.png")
# print(json.dumps(info, indent=2))